# CRIVO — Pensador v2: pensar até a resposta parar de mudar e buscar a prova com o verificador

A v1 (13/10) foi **reprovada**: exato 84,5% no blind (meta 95%), 57,9% em provas longas e 64,6% em problemas grandes (metas 80%). Ela acertava o veredito (97–100%), mas errava a prova mínima, e pensar 16 voltas não era melhor que 8 (`experimentos/pensador_20261012/RESULTADO.md`).

**A v2** (`pensador_v2.py`, `treinar_pensador_v2.py`), com os **mesmos conjuntos congelados e as mesmas metas**:
- **voltas sorteadas no treino** (4 a 16 por lote) e **parada adaptativa** no uso: pensa até a decisão ficar igual em 3 voltas seguidas, de 4 até 32 voltas;
- **busca conferida da prova**: a rede ordena as premissas pela confiança, e o verificador do CRIVO procura nas 10 primeiras o menor conjunto que basta. O verificador só aceita prova verdadeira, então a busca não cria afirmação errada;
- rede um pouco maior (d=192), prova com peso 3 na perda, 30 mil passos.

**Registrado antes de rodar:** o resultado mostra três contas (rede como na v1, adaptativa, adaptativa com busca). **A aprovação usa a adaptativa com busca**, que é a que rodaria no CRIVO. O resultado grava `ativo_no_chat: false`.

**Como rodar:** Ambiente de execução → **GPU (T4 ou L4)** → Executar tudo. Leva uns 40–70 min. Tudo fica em `MyDrive/CRIVO/pensador-20261012/v2_<data>/`.

In [ ]:
# Célula 0 — GPU e Drive
import os, sys, json, subprocess, hashlib, shutil
from pathlib import Path
from datetime import datetime
import torch
if not torch.cuda.is_available():
    raise SystemExit("ERRO: GPU não disponível. Ambiente de execução → Alterar tipo de ambiente → GPU.")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch", torch.__version__, flush=True)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PASSOS = 30000        # passos de treino do pensador v2
VOLTAS = 8            # quantas vezes o bloco é aplicado (pensar mais = mais voltas)
MAX_HORAS = 3.0

RAIZ = Path("/content/drive/MyDrive/CRIVO/pensador-20261012")
SAIDA = RAIZ / datetime.now().strftime("v2_%Y%m%d_%H%M%S")   # nunca sobrescreve
SAIDA.mkdir(parents=True, exist_ok=True)
print("Saída:", SAIDA)


In [ ]:
# Célula 1 — código e conjuntos congelados
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
EXP = REPO / "experimentos" / "pensador_20261012"
ESPERADO = "20deb41adcb704e791027ba5939feb253f6cbd59943de633389eee841500c3df"
obtido = hashlib.sha256((EXP / "conjuntos_pensador.json").read_bytes()).hexdigest()
assert obtido == ESPERADO, "conjuntos congelados diferentes do esperado: " + obtido
(SAIDA / "commit.txt").write_text(COMMIT + "\n")
print("commit", COMMIT, "| conjuntos conferidos", ESPERADO[:12])


In [ ]:
# Célula 2 — treino do pensador v2 (escolha pelo dev; blind só no fim)
def rodar(destino, voltas):
    destino.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "-u", str(EXP / "treinar_pensador_v2.py"), "--saida", str(destino),
           "--passos", str(PASSOS), "--voltas", str(voltas), "--processos", "2", "--max-horas", str(MAX_HORAS)]
    with subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for linha in p.stdout:
            print(linha, end="", flush=True)
    if p.returncode:
        raise SystemExit("treino falhou (código %d)" % p.returncode)
    return json.loads((destino / "resultado_pensador_v2.json").read_text())

v2 = rodar(SAIDA / "v2", VOLTAS)


In [ ]:
# Célula 3 — resumo: as três contas (a aprovação usa "adaptativo_busca", registrado antes)
V1 = {"blind_iid": 0.845, "ood_profundidade": 0.579, "ood_tamanho": 0.646}
print("parâmetros", v2["parametros"], "| passo escolhido", v2["passo_escolhido"])
for nome in ("blind_iid", "ood_profundidade", "ood_tamanho"):
    print("\n" + nome, "(v1: exato %.1f%%)" % (100 * V1[nome]))
    for conta in ("rede", "adaptativo", "adaptativo_busca"):
        m = v2["avaliacao"][nome][conta]
        print("  %-17s exato %5.1f%%  status %5.1f%%  erradas(rede) %4.1f%%  cobertura segura %5.1f%%  voltas %.1f" % (
            conta, 100 * m["exato"], 100 * m["status_acc"], 100 * m["afirmacoes_erradas_brutas"],
            100 * m["politica_cobertura"], m["voltas_media"]))
print("\ninvariância", round(100 * v2["invariancia_status"], 1), "% | gates", v2["gates"])
print("APROVADO:", v2["aprovado"], "| ativo_no_chat:", v2["ativo_no_chat"])
print("Arquivos em", SAIDA, "— avise o assistente.")
